# C3 — Dự báo Hoạt động Đại lý (Churn Prediction)
**DATA EXPLORERS 2026 | Xe đạp Thống Nhất**

Dự báo đại lý nào có nguy cơ **ngừng mua hàng (churn)** trong Q2/2026:
- Xây dựng đặc trưng **RFM** (Recency, Frequency, Monetary) cho từng đại lý
- Thêm đặc trưng bổ sung: số đơn 3 tháng gần nhất
- Huấn luyện **XGBoost Classifier** với tham số tối ưu
- Đánh giá mô hình qua **AUC-ROC, Confusion Matrix, Classification Report**
- Xuất **danh sách đại lý rủi ro cao** để có kế hoạch chăm sóc

**Thứ tự chạy:** Run All (tuần tự từ trên xuống)

## 1. Import Thư viện
> Nhập toàn bộ thư viện cần thiết. Chạy cell này đầu tiên.

In [ ]:
# ── Thư viện chuẩn ──────────────────────────────────────────────────────────
import os
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ── Mô hình phân loại ───────────────────────────────────────────────────────
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    roc_auc_score,
    classification_report,
    confusion_matrix,
    RocCurveDisplay,
)
from sklearn.pipeline import Pipeline

print("✅ Import thư viện thành công")

## 2. Wrangle Data (Tải & Chuẩn hóa Dữ liệu)
> Tải dữ liệu giao dịch đầy đủ. Đây là nguồn dữ liệu chính để xây dựng đặc trưng RFM cho từng đại lý.
>
> **Lưu ý:** ~24% đơn hàng không gắn customer_code — đây là đơn lẻ (walk-in), không phải đại lý thường xuyên. Sẽ được xử lý ở bước Data Checks.

In [ ]:
# ── Tải dữ liệu giao dịch ────────────────────────────────────────────────────
df_fact = pd.read_csv("data/fact_sales_full.csv", parse_dates=["order_date"])

print(f"Shape            : {df_fact.shape}")
print(f"Khoảng thời gian : {df_fact.order_date.min().strftime('%m/%Y')} → {df_fact.order_date.max().strftime('%m/%Y')}")
df_fact.head()

## 3. Kiểm tra Dữ liệu (Data Checks)
> Kiểm tra giá trị thiếu, thống kê đại lý và làm sạch dữ liệu.

In [ ]:
# ── Kiểu dữ liệu ─────────────────────────────────────────────────────────────
print(f"Shape: {df_fact.shape}")
print(f"\nKiểu dữ liệu:")
print(df_fact.dtypes)

In [ ]:
# ── Kiểm tra giá trị thiếu ───────────────────────────────────────────────────
missing = df_fact.isnull().sum()
print("Giá trị thiếu:")
print(missing[missing > 0])
print(f"\n⚠️  customer_code thiếu {df_fact.customer_code.isnull().mean()*100:.1f}% — đơn không có đại lý cố định")

In [ ]:
# ── Thống kê và làm sạch ─────────────────────────────────────────────────────
# Loại bỏ dòng thiếu thông tin đại lý
df_clean = df_fact.dropna(subset=["customer_code", "customer_name"]).copy()
df_clean["line_total"] = df_clean["line_total"].fillna(0)
df_clean["quantity"]   = df_clean["quantity"].fillna(0)

print(f"Tổng đơn hàng    : {len(df_fact):,}")
print(f"Sau clean        : {len(df_clean):,} đơn ({len(df_clean)/len(df_fact)*100:.1f}%)")
print(f"Số đại lý        : {df_clean.customer_code.nunique():,}")
print(f"Tổng doanh thu   : {df_clean.line_total.sum()/1e9:.1f} tỷ VND")

## 4. Xây dựng Đặc trưng RFM (Feature Engineering)
> RFM là framework phân tích hành vi khách hàng:
> - **Recency (R):** Số ngày kể từ lần mua gần nhất
> - **Frequency (F):** Tổng số đơn hàng đã đặt
> - **Monetary (M):** Tổng giá trị đơn hàng (VND)
> - **last3m_orders:** Số đơn trong 3 tháng gần nhất (đặc trưng bổ sung)
>
> Đại lý có **Recency cao** (lâu không mua) và **Frequency/Monetary thấp** → rủi ro churn cao.

In [ ]:
# ── Ngày tham chiếu = ngày cuối cùng có dữ liệu ─────────────────────────────
reference_date = df_clean["order_date"].max()
print(f"Ngày tham chiếu: {reference_date.strftime('%d/%m/%Y')}")

# ── Tính RFM ─────────────────────────────────────────────────────────────────
rfm = df_clean.groupby("customer_code").agg(
    recency   = ("order_date", lambda x: (reference_date - x.max()).days),
    frequency = ("so_number",  "nunique"),
    monetary  = ("line_total", "sum"),
).reset_index()

# ── Đặc trưng bổ sung: số đơn 3 tháng gần nhất ──────────────────────────────
cutoff_3m = reference_date - pd.DateOffset(months=3)
last3m = (
    df_clean[df_clean["order_date"] >= cutoff_3m]
    .groupby("customer_code")["so_number"]
    .nunique()
    .reset_index()
    .rename(columns={"so_number": "last3m_orders"})
)
rfm = rfm.merge(last3m, on="customer_code", how="left")
rfm["last3m_orders"] = rfm["last3m_orders"].fillna(0).astype(int)

print(f"\nRFM shape: {rfm.shape}")
rfm.head()

In [ ]:
# ── Phân phối RFM ────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, col, title, unit in zip(
    axes,
    ["recency", "frequency", "monetary"],
    ["Recency (ngày kể từ mua cuối)", "Frequency (số đơn)", "Monetary (VND)"],
    ["ngày", "đơn", "VND"],
):
    ax.hist(rfm[col], bins=25, alpha=0.8, edgecolor="white")
    med = rfm[col].median()
    ax.axvline(med, linestyle="--", linewidth=1.5, label=f"Trung vị: {med:.0f} {unit}")
    ax.set_title(title, fontsize=11, fontweight="bold")
    ax.set_xlabel(unit.capitalize())
    ax.set_ylabel("Số đại lý")
    ax.legend(fontsize=9)
    ax.grid(axis="y", alpha=0.4)

fig.suptitle("Phân phối Đặc trưng RFM — Đại lý Thống Nhất Bike", fontsize=13, fontweight="bold")
fig.tight_layout()
plt.show()

## 5. Gán nhãn Churn (Labeling)
> Định nghĩa churn dựa trên Recency: đại lý không mua trong 90 ngày = churn.
> Đây là ngưỡng hợp lý cho thị trường xe đạp B2B (chu kỳ mua theo quý).

In [ ]:
# ── Gán nhãn Churn ───────────────────────────────────────────────────────────
CHURN_THRESHOLD = 90  # ngày

rfm["churn"] = (rfm["recency"] > CHURN_THRESHOLD).astype(int)

churn_rate = rfm["churn"].mean()
print(f"Ngưỡng churn     : recency > {CHURN_THRESHOLD} ngày")
print(f"Đại lý CHURN     : {rfm.churn.sum():,} / {len(rfm):,}  ({churn_rate*100:.1f}%)")
print(f"Đại lý ACTIVE    : {(rfm.churn==0).sum():,} ({(1-churn_rate)*100:.1f}%)")

In [ ]:
# ── Phân bố churn theo Recency × Monetary ────────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 5))
churn0 = rfm[rfm.churn == 0]
churn1 = rfm[rfm.churn == 1]

ax.scatter(churn0.recency, churn0.monetary / 1e6, alpha=0.5, s=30, label="Active")
ax.scatter(churn1.recency, churn1.monetary / 1e6, alpha=0.5, s=30, marker="x", label="Churn")
ax.axvline(CHURN_THRESHOLD, linestyle="--", alpha=0.7, linewidth=2,
           label=f"Ngưỡng {CHURN_THRESHOLD} ngày")
ax.set_xlabel("Recency (ngày)")
ax.set_ylabel("Monetary (triệu VND)")
ax.set_title("Phân bố Đại lý — Active vs Churn", fontsize=12, fontweight="bold")
ax.legend()
ax.grid(alpha=0.3)
fig.tight_layout()
plt.show()

## 6. Xây dựng Pipeline Mô hình
> Chuẩn bị dữ liệu đầu vào: chuẩn hóa đặc trưng (StandardScaler) và tách tập train/test.
> Tải tham số XGBoost tốt nhất từ bước Parameter Tuning.

In [ ]:
# ── Tải tham số tốt nhất ─────────────────────────────────────────────────────
params_clf = pd.read_csv("best_params/best_params_classifier.csv", index_col=0)
n_estimators = int(params_clf.loc["n_estimators"][0])
max_depth    = int(params_clf.loc["max_depth"][0])
lr           = float(params_clf.loc["learning_rate"][0])
subsample    = float(params_clf.loc["subsample"][0])

print("Tham số XGBoost tốt nhất:")
print(f"  n_estimators  = {n_estimators}")
print(f"  max_depth     = {max_depth}")
print(f"  learning_rate = {lr}")
print(f"  subsample     = {subsample}")

In [ ]:
# ── Chuẩn bị ma trận đặc trưng và nhãn ──────────────────────────────────────
FEATURE_COLS = ["recency", "frequency", "monetary", "last3m_orders"]

X = rfm[FEATURE_COLS].values
y = rfm["churn"].values

# ── Chuẩn hóa đặc trưng ──────────────────────────────────────────────────────
scaler  = StandardScaler()
X_scaled = scaler.fit_transform(X)

# ── Tách tập train / test ─────────────────────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train : {len(X_train)} mẫu  ({len(X_train)/len(X)*100:.0f}%)")
print(f"Test  : {len(X_test)}  mẫu  ({len(X_test)/len(X)*100:.0f}%)")
print(f"Tỷ lệ churn trong train: {y_train.mean()*100:.1f}%")
print(f"Tỷ lệ churn trong test : {y_test.mean()*100:.1f}%")

## 7. Huấn luyện Mô hình XGBoost
> Huấn luyện XGBoost Classifier với tham số tốt nhất.
> XGBoost phù hợp hơn LSTM cho bài toán phân loại churn dạng tabular (RFM features).

In [ ]:
# ── Huấn luyện XGBoost ───────────────────────────────────────────────────────
clf = XGBClassifier(
    n_estimators  = n_estimators,
    max_depth     = max_depth,
    learning_rate = lr,
    subsample     = subsample,
    eval_metric   = "logloss",
    random_state  = 42,
    verbosity     = 0,
)
clf.fit(X_train, y_train)
print("✅ XGBoost đã huấn luyện xong")

## 8. Đánh giá Mô hình
> Đánh giá toàn diện: Cross-Validation AUC, Classification Report, Confusion Matrix và ROC Curve.

In [ ]:
# ── Cross-Validation AUC-ROC ──────────────────────────────────────────────────
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
auc_scores = cross_val_score(clf, X_scaled, y, cv=cv, scoring="roc_auc")

print("AUC-ROC (5-fold Stratified CV):")
print(f"  Mean   : {auc_scores.mean():.4f}")
print(f"  Std    : {auc_scores.std():.4f}")
print(f"  Scores : {np.round(auc_scores, 4)}")

In [ ]:
# ── Dự báo trên tập test ─────────────────────────────────────────────────────
y_pred = clf.predict(X_test)
y_prob = clf.predict_proba(X_test)[:, 1]

print(f"AUC-ROC (test set): {roc_auc_score(y_test, y_prob):.4f}")
print()
print("Classification Report:")
print(classification_report(y_test, y_pred, target_names=["Active", "Churn"]))

In [ ]:
# ── Confusion Matrix + ROC Curve ─────────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
im = ax1.imshow(cm, interpolation="nearest", aspect="auto")
ax1.set_title("Confusion Matrix", fontsize=12, fontweight="bold")
ax1.set_xlabel("Dự báo")
ax1.set_ylabel("Thực tế")
ax1.set_xticks([0, 1])
ax1.set_yticks([0, 1])
ax1.set_xticklabels(["Active", "Churn"])
ax1.set_yticklabels(["Active", "Churn"])
for i in range(2):
    for j in range(2):
        ax1.text(j, i, str(cm[i, j]), ha="center", va="center",
                 fontsize=16, fontweight="bold",
                 color="white" if cm[i, j] > cm.max() / 2 else "black")
fig.colorbar(im, ax=ax1)

# ROC Curve
RocCurveDisplay.from_predictions(y_test, y_prob, ax=ax2, name="XGBoost")
ax2.set_title("ROC Curve — XGBoost Churn Model", fontsize=12, fontweight="bold")
ax2.plot([0, 1], [0, 1], "k--", alpha=0.5)
ax2.grid(alpha=0.3)

fig.tight_layout()
plt.show()

In [ ]:
# ── Feature Importance ───────────────────────────────────────────────────────
fi = pd.Series(clf.feature_importances_, index=FEATURE_COLS).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(fi.index, fi.values, alpha=0.85)
for i, v in enumerate(fi.values):
    ax.text(v + 0.005, i, f"{v:.3f}", va="center", fontsize=10, fontweight="bold")
ax.set_xlabel("Importance Score (Gain)")
ax.set_title("Feature Importance — XGBoost Churn Model", fontsize=12, fontweight="bold")
ax.grid(axis="x", alpha=0.4)
fig.tight_layout()
plt.show()

print(f"Đặc trưng quan trọng nhất: {fi.idxmax()} ({fi.max():.3f})")

## 9. Danh sách Đại lý Rủi ro Cao
> Áp dụng mô hình lên toàn bộ đại lý để lấy xác suất churn và phân loại mức độ rủi ro.
> Kết quả này giúp đội Sales ưu tiên chăm sóc đúng đối tượng.

In [ ]:
# ── Dự báo xác suất churn toàn bộ đại lý ────────────────────────────────────
rfm["churn_prob"] = clf.predict_proba(X_scaled)[:, 1]
rfm["churn_pred"] = clf.predict(X_scaled)
rfm["risk_level"] = pd.cut(
    rfm["churn_prob"],
    bins=[0, 0.3, 0.6, 1.0],
    labels=["Thấp", "Trung bình", "Cao"],
)

# ── Gắn tên đại lý ───────────────────────────────────────────────────────────
name_map = (
    df_clean.drop_duplicates("customer_code")[["customer_code", "customer_name"]]
)
rfm_named = rfm.merge(name_map, on="customer_code", how="left")

# ── Top 20 đại lý rủi ro cao nhất ────────────────────────────────────────────
high_risk = (
    rfm_named[rfm_named["risk_level"] == "Cao"]
    .sort_values("churn_prob", ascending=False)
    .head(20)
)
print(f"Tổng đại lý rủi ro CAO: {(rfm['risk_level']=='Cao').sum()}")
high_risk[["customer_code", "customer_name", "recency", "frequency", "monetary", "churn_prob"]].head(10)

In [ ]:
# ── Biểu đồ phân bố xác suất churn + phân loại rủi ro ───────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Phân bố xác suất
ax1.hist(rfm["churn_prob"], bins=25, alpha=0.8, edgecolor="white")
ax1.axvline(0.3, linestyle="--", linewidth=2, label="Ngưỡng Thấp/Trung bình (0.3)")
ax1.axvline(0.6, linestyle=":",  linewidth=2, label="Ngưỡng Trung bình/Cao (0.6)")
ax1.set_xlabel("Xác suất Churn")
ax1.set_ylabel("Số đại lý")
ax1.set_title("Phân bố Xác suất Churn — Toàn bộ Đại lý", fontsize=12, fontweight="bold")
ax1.legend(fontsize=9)
ax1.grid(axis="y", alpha=0.4)

# Phân loại rủi ro
risk_counts = rfm["risk_level"].value_counts().reindex(["Thấp", "Trung bình", "Cao"])
ax2.bar(risk_counts.index, risk_counts.values, alpha=0.85)
for i, (label, v) in enumerate(risk_counts.items()):
    ax2.text(i, v + 1, str(v), ha="center", va="bottom", fontsize=12, fontweight="bold")
ax2.set_xlabel("Mức độ rủi ro")
ax2.set_ylabel("Số đại lý")
ax2.set_title("Phân loại Đại lý theo Mức độ Rủi ro Churn", fontsize=12, fontweight="bold")
ax2.grid(axis="y", alpha=0.4)

fig.tight_layout()
plt.show()

## 10. Xuất Kết quả
> Lưu danh sách đại lý churn và kết quả phân tích vào thư mục `output/`.

In [ ]:
# ── Chuẩn bị dataframe xuất ──────────────────────────────────────────────────
rfm_out = rfm_named[[
    "customer_code", "customer_name",
    "recency", "frequency", "monetary", "last3m_orders",
    "churn_prob", "churn_pred", "risk_level",
]].sort_values("churn_prob", ascending=False)

# ── Lưu file ─────────────────────────────────────────────────────────────────
os.makedirs("output", exist_ok=True)
rfm_out.to_csv("output/predictions_churn.csv", index=False)

print("✅ Đã lưu kết quả vào output/")
print("   → predictions_churn.csv")
print(f"\nPreview (top 5 rủi ro cao):")
print(rfm_out.head(5).to_string(index=False))

In [ ]:
# ── Tóm tắt kết quả cuối cùng ────────────────────────────────────────────────
auc_final = roc_auc_score(y_test, y_prob)
risk_summary = rfm["risk_level"].value_counts().reindex(["Thấp", "Trung bình", "Cao"])

print("=" * 60)
print("   KẾT QUẢ C3 — DỰ BÁO CHURN ĐẠI LÝ Q2/2026")
print("=" * 60)
print(f"\n  Tổng đại lý phân tích   : {len(rfm):,}")
print(f"  Đại lý Churn (thực tế) : {rfm.churn.sum():,}  ({rfm.churn.mean()*100:.1f}%)")
print(f"  AUC-ROC (5-fold CV)    : {auc_scores.mean():.4f}")
print(f"  AUC-ROC (test set)     : {auc_final:.4f}")
print(f"  Đặc trưng quan trọng   : {fi.idxmax()}  ({fi.max():.3f})")
print(f"\n  Phân loại rủi ro:")
for level in ["Thấp", "Trung bình", "Cao"]:
    cnt = risk_summary[level]
    pct = cnt / len(rfm) * 100
    print(f"    {level:<12}: {cnt:,} đại lý  ({pct:.1f}%)")
print("\n" + "=" * 60)